# LS-LLM: longer training runs on a Colab GPU
The laptop (M1, 8 GB) trains the Tier A model at about 70,000 tokens per second, so the runs of the report are short
(10,000 steps of 64 x 256 tokens). This notebook runs the same code with the same data on a Colab GPU for longer.
Upload `lsllm_colab_bundle.zip` (made by `tools/make_colab_bundle.py`) when asked.
Runtime > Change runtime type > GPU before you start.

In [ ]:
from google.colab import files
up = files.upload()
!unzip -q -o lsllm_colab_bundle.zip
!pip -q install sentencepiece
!nvidia-smi | head -15

## Tier A float model: the shape of the report, 8 times more tokens

In [ ]:
!python -m lsllm.train.run --out runs/f/A_long --data data/tok/f512v1 --device cuda --batch-size 128 --prompt-weight 0.1 --eval-every 2000 --n-layers 2 --dim 160 --n-heads 4 --n-kv-heads 1 --ffn relu --hidden-dim 219 --steps 40000 --lr 1e-3

## Tier B float model

In [ ]:
!python -m lsllm.train.run --out runs/f/B_long --data data/tok/f512v1 --device cuda --batch-size 128 --prompt-weight 0.1 --eval-every 2000 --n-layers 4 --dim 192 --n-heads 4 --n-kv-heads 1 --ffn relu --hidden-dim 510 --steps 40000 --lr 6e-4

## Quantization-aware training (the forward pass is the arithmetic of the LS CPU, bit for bit)

In [ ]:
!python -m lsllm.qat.train --float runs/f/A_long/ckpt.pt --data data/tok/f512v1 --out runs/qat/A_long --steps 6000 --kd-weight 0.5 --batch-size 32 --device cuda
!python -m lsllm.qat.train --float runs/f/B_long/ckpt.pt --data data/tok/f512v1 --out runs/qat/B_long --steps 6000 --kd-weight 0.5 --batch-size 32 --device cuda

## Download the results
Copy the downloaded `runs/` folders into the project on the laptop, then run `tools/finalize.py` for each model
(benchmarks in the sandbox, export, speed documents) and `tools/report.py`.

In [ ]:
!zip -q -r runs.zip runs
files.download('runs.zip')